# E-Commerce Search & Conversion Funnel Analysis
### A College-Level Product Analytics Project using SQL & Python

This notebook analyzes an apparel marketplace dataset to understand how customer search behavior affects conversion throughput. We investigate why specific, multi-attribute searches fail to return products, and prototype a simple automated query-relaxation fallback.

## 1. Setup & Database Connection

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

# Add project root to sys.path
sys.path.insert(0, '..')

con = duckdb.connect('../data/ecommerce_analytics.duckdb', read_only=True)
print("DuckDB Connected!")
con.execute("SHOW TABLES").df()

## 2. Funnel Analysis: Search-Engaged vs Browse-Only Shoppers
We evaluate conversion progression: Sessions -> PDP View -> Cart Add -> Completed Order.

In [ ]:
with open('../sql/01_funnel_analysis.sql', 'r') as f:
    sql_funnel = f.read()

# Execute second query in the file (funnel comparison)
queries = [q.strip() for q in sql_funnel.split(';') if q.strip()]
df_funnel = con.execute(queries[1]).df()
df_funnel

**Key Finding**: Search-engaged shoppers convert at **11.92%**, more than 2x higher than browse-only shoppers (**5.16%**). Search is clearly the highest-intent discovery surface on the platform.

## 3. Search Discovery Breakdown on Multi-Attribute Queries
Next, we segment queries by word count (1–3 tokens vs 4+ tokens) to see where search fails.

In [ ]:
with open('../sql/02_search_analysis.sql', 'r') as f:
    sql_search = f.read()

queries = [q.strip() for q in sql_search.split(';') if q.strip()]
df_search = con.execute(queries[0]).df()
df_search

In [ ]:
# Eligible low-result cohort (<3 results on 4+ tokens)
df_eligible = con.execute(queries[1]).df()
df_eligible

**Problem Identified**: Specific queries (4+ words) represent **33.85%** of all searches, but suffer an **8.23% Zero-Result Rate** (4.6x higher than head queries) and a **44.39% manual reformulation rate**. Among the 941 searches that returned fewer than 3 results, Search-to-PDP Click-Through Rate collapsed to **3.08%**.

## 4. Query Relaxation Prototype
We test our simple Python prototype (`src/search.py`). When a 4+ token query yields < 3 results, it uses an **iterative relaxation approach**: testing 1-token removal first, then combinations of 2–3 modifiers if needed, while always protecting core category nouns and enforcing at least 2 remaining tokens.


In [ ]:
from src.search import load_catalog, relaxed_search

catalog = load_catalog('../data/ecommerce_analytics.duckdb')

test_queries = [
    'slim fit black dresses XL',
    'vintage black jeans L',
    'breathable black jeans M',
    'shoes'
]

for q in test_queries:
    res = relaxed_search(q, catalog)
    print(f"Query: '{q}' -> Relaxed: {res['is_relaxed']} | New Query: '{res['relaxed_query']}' | Results: {res['result_count']}")

## 5. Business Impact Estimation
We estimate downstream conversion and GMV impact if query relaxation improves Search-to-PDP CTR on the 941 eligible searches.

In [ ]:
from src.business_impact import get_baseline_metrics, estimate_scenario

baseline = get_baseline_metrics('../data/ecommerce_analytics.duckdb')
scenarios = [
    estimate_scenario(baseline, ctr_lift=0.010, scenario_name="Scenario A (+1.0 pp CTR Lift)"),
    estimate_scenario(baseline, ctr_lift=0.035, scenario_name="Scenario B (+3.5 pp CTR Lift)"),
]
pd.DataFrame(scenarios)

## 6. Product Recommendation
- **Sizing**: Improving CTR by +3.5 pp produces **+$297.32 in 60-day GMV** (~**+$1,808.69 annualized**).
- **Capital Discipline**: At current boutique volume (~16 searches/day), the standalone revenue is modest. Therefore, we recommend running a lightweight A/B test first to validate customer conversion before investing in complex search infrastructure.